# MAT 504 — Computer Modelling in Materials Science
## Week 4 Hands-on — When Can I Trust an MD Trajectory?

### Objective

In Week 4, the question is no longer only **how to generate a molecular-dynamics trajectory**, but **how to decide whether the trajectory represents the intended physical conditions and whether an observable calculated from it can be trusted**.

In this exercise you will use the same small two-dimensional Lennard–Jones system as in Week 3 to:

1. compare **NVE** and **NVT** dynamics;
2. distinguish **equilibration** from **production**;
3. calculate an equilibrium average from the production region;
4. examine the **autocorrelation of a specific observable**, \(A(t)=U(t)/N\);
5. use **block averaging** to estimate uncertainty from correlated data.

The central sequence is:

**ensemble → equilibration → production → observable → correlation → uncertainty → trust**.


## Computational setup

The low-level molecular-dynamics machinery is supplied. You are **not** expected to implement a thermostat, periodic boundaries, or the integrator in this exercise. Read the helper functions so that you understand what is being done, but focus your work on the **physical interpretation and analysis**.

We use a 2D Lennard–Jones fluid in reduced units,

\[
\epsilon=\sigma=m=k_B=1.
\]

The helper uses periodic boundaries with the minimum-image convention. Because this is a very small teaching system, it evaluates all particle pairs directly; large production simulations would normally use additional efficiency machinery such as cutoffs and neighbor lists.

For NVT, the supplied helper uses a simple **Andersen thermostat**. The NVT trajectory starts deliberately below the target temperature so that the equilibration stage is visible.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=5, suppress=True)

## Supplied MD and analysis helper functions

Run this cell without changing it. You are encouraged to read the comments.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def initialize_lattice(n_particles, box_length):
    """Place particles on a square lattice."""
    n_side = int(np.ceil(np.sqrt(n_particles)))
    spacing = box_length / n_side
    coordinates = []

    for iy in range(n_side):
        for ix in range(n_side):
            if len(coordinates) >= n_particles:
                break
            coordinates.append([(ix + 0.5) * spacing,
                                (iy + 0.5) * spacing])
        if len(coordinates) >= n_particles:
            break

    return np.array(coordinates, dtype=float)


def lj_forces(positions_wrapped, box_length):
    """Return Lennard-Jones forces and potential energy in reduced units.

    This small educational system evaluates all particle pairs. Periodic
    boundaries are handled with the minimum-image convention.
    """
    r = positions_wrapped

    # Pair displacement r_i - r_j
    dr = r[:, None, :] - r[None, :, :]
    dr -= box_length * np.rint(dr / box_length)

    r2 = np.sum(dr**2, axis=2)
    np.fill_diagonal(r2, np.inf)

    inv_r2 = 1.0 / r2
    inv_r6 = inv_r2**3

    # F_ij = 24 [2 r^-14 - r^-8] r_ij
    prefactor = 24.0 * inv_r2 * (2.0 * inv_r6**2 - inv_r6)
    prefactor[np.isinf(r2)] = 0.0
    forces = np.sum(prefactor[:, :, None] * dr, axis=1)

    # Pair potential; factor 1/2 removes double counting
    pair_u = 4.0 * (inv_r6**2 - inv_r6)
    np.fill_diagonal(pair_u, 0.0)
    potential_energy = 0.5 * np.sum(pair_u)

    return forces, potential_energy


def instantaneous_temperature(velocities):
    """Return the instantaneous reduced temperature and kinetic energy."""
    kinetic_energy = 0.5 * np.sum(velocities**2)
    degrees_of_freedom = velocities.size
    temperature = 2.0 * kinetic_energy / degrees_of_freedom
    return temperature, kinetic_energy


def run_lj_md(
    ensemble='NVE',
    target_temperature=1.0,
    initial_temperature=0.5,
    dt=0.005,
    total_time=60.0,
    n_particles=36,
    density=0.65,
    sample_dt=0.05,
    seed=504,
    collision_rate=1.0,
):
    """Run a small 2D Lennard-Jones simulation with velocity Verlet.

    NVE: ordinary conservative velocity-Verlet dynamics.
    NVT: the same dynamics plus a simple Andersen thermostat. Each particle
         has a small probability collision_rate*dt of being reassigned a
         velocity drawn from the target thermal distribution.

    The thermostat implementation is supplied for this exercise; students are
    not expected to derive or modify it.
    """
    ensemble = ensemble.upper()
    if ensemble not in {'NVE', 'NVT'}:
        raise ValueError("ensemble must be 'NVE' or 'NVT'")

    box_length = np.sqrt(n_particles / density)
    positions = initialize_lattice(n_particles, box_length)
    positions_unwrapped = positions.copy()

    rng = np.random.default_rng(seed)

    velocities = rng.normal(size=(n_particles, 2))
    velocities -= velocities.mean(axis=0)

    current_temperature, _ = instantaneous_temperature(velocities)
    velocities *= np.sqrt(initial_temperature / current_temperature)

    forces, potential_energy = lj_forces(positions, box_length)

    n_steps = int(round(total_time / dt))
    sample_every = max(1, int(round(sample_dt / dt)))

    times = []
    temperatures = []
    potential_per_particle = []
    kinetic_per_particle = []
    total_per_particle = []
    trajectory = []

    for step in range(n_steps + 1):
        if step % sample_every == 0:
            temperature, kinetic_energy = instantaneous_temperature(velocities)
            times.append(step * dt)
            temperatures.append(temperature)
            potential_per_particle.append(potential_energy / n_particles)
            kinetic_per_particle.append(kinetic_energy / n_particles)
            total_per_particle.append((kinetic_energy + potential_energy) / n_particles)
            trajectory.append(positions_unwrapped.copy())

        if step == n_steps:
            break

        # Velocity-Verlet integration
        velocities += 0.5 * dt * forces
        positions_unwrapped += dt * velocities
        positions = np.mod(positions_unwrapped, box_length)

        new_forces, new_potential_energy = lj_forces(positions, box_length)
        velocities += 0.5 * dt * new_forces

        # Simple Andersen thermostat for NVT
        if ensemble == 'NVT':
            collisions = rng.random(n_particles) < collision_rate * dt
            n_collisions = int(collisions.sum())
            if n_collisions:
                velocities[collisions] = rng.normal(
                    scale=np.sqrt(target_temperature),
                    size=(n_collisions, 2),
                )

        forces = new_forces
        potential_energy = new_potential_energy

    return {
        'ensemble': ensemble,
        'time': np.array(times),
        'temperature': np.array(temperatures),
        'potential_per_particle': np.array(potential_per_particle),
        'kinetic_per_particle': np.array(kinetic_per_particle),
        'total_per_particle': np.array(total_per_particle),
        'trajectory': np.array(trajectory),
        'box_length': box_length,
    }


def normalized_autocorrelation(values, max_lag=None):
    """Normalized autocorrelation C(k) of one scalar observable time series."""
    x = np.asarray(values, dtype=float)
    x = x - x.mean()

    if max_lag is None:
        max_lag = len(x) - 1

    corr = np.correlate(x, x, mode='full')[len(x)-1:]
    normalization = np.arange(len(x), 0, -1)
    corr = corr / normalization
    corr = corr / corr[0]
    return corr[:max_lag + 1]


def block_means(values, n_blocks=10):
    """Split a time series into equal contiguous blocks and return block means."""
    values = np.asarray(values, dtype=float)
    n_per_block = len(values) // n_blocks
    trimmed = values[:n_per_block * n_blocks]
    return trimmed.reshape(n_blocks, n_per_block).mean(axis=1)


## Part 1 — NVE versus NVT

We will run two simulations from the same initial coordinates and velocities:

- **NVE:** no thermostat; total energy should remain approximately constant.
- **NVT:** target temperature \(T=1.0\); the thermostat exchanges energy with the system.

Both runs start at \(T=0.5\).

### Question 1 — predict before running

**a.** In which ensemble do you expect the **total energy** to remain approximately constant?

**b.** In which ensemble do you expect the **temperature** to relax toward a chosen target value?

**c.** In NVT, should the instantaneous temperature equal the target value exactly at every stored frame?

**Answer:**

_Write your answer here._


In [ ]:
# Simulation parameters
params = dict(
    initial_temperature=0.5,
    dt=0.005,
    total_time=60.0,
    n_particles=36,
    density=0.65,
    sample_dt=0.05,
    seed=504,
)

# TODO 1: run the NVE simulation
run_nve = ...

# TODO 2: run the NVT simulation with target_temperature=1.0
run_nvt = ...


### Task 1.1 — Compare temperature and total energy

Plot the NVE and NVT temperature histories on one graph. Then plot total energy per particle on a second graph.

In [ ]:
plt.figure(figsize=(8, 4.2))
# TODO: plot temperature versus time for both NVE and NVT

plt.axhline(1.0, linestyle='--', linewidth=1, label='NVT target T')
plt.xlabel('Time (reduced units)')
plt.ylabel('Temperature')
plt.title('NVE versus NVT: temperature')
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4.2))
# TODO: plot total energy per particle versus time for both NVE and NVT

plt.xlabel('Time (reduced units)')
plt.ylabel('Total energy per particle')
plt.title('NVE versus NVT: total energy')
plt.legend()
plt.tight_layout()
plt.show()


### Question 2

**a.** Which feature of the NVE plots indicates that the trajectory is behaving as expected for conservative dynamics?

**b.** Why is total energy **not** expected to remain constant in the NVT trajectory?

**c.** Does the NVT temperature fluctuate around the target, or is it fixed exactly at the target? Why is that distinction important?

**Answer:**

_Write your answer here._

## Part 2 — Is the NVT trajectory equilibrated?

A simulation does not become production data simply because it has been running for some time. We need to ask whether the system has lost the strong influence of its arbitrary initial state.

For this exercise, use

\[
t_{\mathrm{eq}}=10
\]

as the proposed end of equilibration.

Plot both the **temperature** and **potential energy per particle** and mark \(t_{\mathrm{eq}}\) with a vertical line.


In [ ]:
t_eq = 10.0

plt.figure(figsize=(8, 4.2))
plt.plot(run_nvt['time'], run_nvt['temperature'])
plt.axhline(1.0, linestyle='--', linewidth=1, label='target T')
plt.axvline(t_eq, linestyle=':', linewidth=1.5, label='proposed end of equilibration')
plt.xlabel('Time (reduced units)')
plt.ylabel('Temperature')
plt.title('NVT temperature during equilibration and production')
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4.2))
# TODO: plot potential energy per particle versus time
# TODO: add a vertical line at t_eq

plt.xlabel('Time (reduced units)')
plt.ylabel('Potential energy per particle')
plt.title('Potential energy during equilibration and production')
plt.legend()
plt.tight_layout()
plt.show()


### Question 3

Based on **both** the temperature and potential-energy plots, is \(t_{\mathrm{eq}}=10\) a reasonable cutoff for this exercise? What changes before and after this time?

Do not claim that a single flat-looking curve proves perfect equilibration; explain only whether this is a reasonable working choice for the present exercise.

**Answer:**

_Write your answer here._

## Part 3 — Production averages

We now use only the NVT data after

\[
t \ge 10
\]

as production data.

Our observable will be the **potential energy per particle**,

\[
A(t)=\frac{U(t)}{N}.
\]

Calculate:

1. the average \(U/N\) using the **entire trajectory**;
2. the average \(U/N\) using **production only**.

The point is not that the two values must be dramatically different. The point is that the production estimate has a clearer statistical meaning because the initial relaxation is excluded.


In [ ]:
# TODO 1: Boolean mask selecting the NVT production region t >= t_eq
production_mask = ...

u_per_particle = run_nvt['potential_per_particle']
u_production = u_per_particle[production_mask]

# TODO 2: calculate the full-trajectory and production-only means
mean_u_all = ...
mean_u_prod = ...

print(f'Mean U/N using full trajectory: {mean_u_all:.5f}')
print(f'Mean U/N using production only: {mean_u_prod:.5f}')
print(f'Difference: {mean_u_all - mean_u_prod:+.5f}')


### Question 4

Why is the production-only average scientifically preferable even if the numerical difference from the full-trajectory average is modest in this particular run?

**Answer:**

_Write your answer here._

## Part 4 — Trajectory frames are correlated

Saving 1,000 frames does **not** imply that we have 1,000 independent samples.

Here the autocorrelation is explicitly the autocorrelation of the scalar observable

\[
A(t)=U(t)/N.
\]

The supplied function `normalized_autocorrelation` calculates the normalized autocorrelation \(C_A(k)\), where \(k\) is the lag measured in stored frames.

A slowly decaying \(C_A(k)\) means that the observable retains memory for many stored frames. A rapid decay means that the stored values become less correlated more quickly.


In [ ]:
sample_interval = run_nvt['time'][1] - run_nvt['time'][0]

# TODO 1: calculate C_A(k) for the production U/N time series.
# Use at most 200 lags.
acf_u = ...

lag_time = np.arange(len(acf_u)) * sample_interval

plt.figure(figsize=(7.5, 4.2))
# TODO 2: plot autocorrelation versus lag_time

plt.axhline(0, linewidth=0.8)
plt.xlabel('Lag time (reduced units)')
plt.ylabel(r'$C_A(k)$')
plt.title(r'Autocorrelation of $A(t)=U(t)/N$')
plt.tight_layout()
plt.show()

# Optional descriptive number: first lag where C_A falls below 1/e
below = np.where(acf_u < np.exp(-1))[0]
if len(below) > 0:
    first_index = below[0]
    print(f'First lag below 1/e: {lag_time[first_index]:.3f} time units')
else:
    print('C_A did not fall below 1/e in the plotted range.')


### Question 5

**a.** What specific quantity is being autocorrelated here?

**b.** What does the decay of this autocorrelation tell you about the independence of neighboring stored values of \(U/N\)?

**c.** Why would simply saving frames more frequently not necessarily give more independent information?

**Answer:**

_Write your answer here._

## Part 5 — Block averaging and uncertainty

To estimate uncertainty from correlated trajectory data, divide the production time series into contiguous blocks that are long compared with the correlation time.

For this exercise, use **10 equal blocks**. For each block, calculate the mean potential energy per particle.

Then compare two uncertainty estimates:

1. **Naive standard error:** treats every stored production frame as independent;
2. **Block standard error:** treats the block means as the approximately independent observations.

For the block means \(\bar A_b\), use

\[
\mathrm{SE}_{\mathrm{block}}
=
\frac{s_{\mathrm{blocks}}}{\sqrt{B}},
\]

where \(B\) is the number of blocks and \(s_{\mathrm{blocks}}\) is the sample standard deviation of the block means.


In [ ]:
n_blocks = 10

# TODO 1: calculate the 10 block means
u_block_means = ...

# Naive SE: assumes every stored production frame is independent
naive_se = np.std(u_production, ddof=1) / np.sqrt(len(u_production))

# TODO 2: block-based SE from the spread of the block means
block_se = ...

print('Block means:')
print(u_block_means)
print(f'\nProduction mean U/N: {np.mean(u_production):.5f}')
print(f'Naive SE:             {naive_se:.5f}')
print(f'Block-based SE:       {block_se:.5f}')

plt.figure(figsize=(8, 4.2))
plt.plot(np.arange(1, n_blocks + 1), u_block_means, marker='o')
plt.axhline(np.mean(u_production), linestyle='--', linewidth=1, label='production mean')
plt.xlabel('Block number')
plt.ylabel('Block mean of U/N')
plt.title('Block means of the production trajectory')
plt.legend()
plt.tight_layout()
plt.show()


### Question 6

**a.** Which uncertainty estimate is larger: the naive standard error or the block-based standard error?

**b.** Why does treating every stored frame as independent tend to underestimate uncertainty when the trajectory is correlated?

**c.** Why must blocks be long compared with the correlation time?

**Answer:**

_Write your answer here._

## Final synthesis

In **3–4 sentences**, explain why the following statement is wrong:

> “My MD simulation ran without crashing and I saved many frames, so the reported average is trustworthy.”

Your answer should mention **ensemble**, **equilibration/production**, **correlation**, and **uncertainty or convergence**.

**Answer:**

_Write your answer here._

## Before you leave

Make sure your notebook contains:

1. your NVE/NVT prediction;
2. the NVE/NVT temperature and total-energy comparison;
3. the NVT equilibration plots;
4. the full-trajectory and production-only averages of \(U/N\);
5. the autocorrelation of \(U/N\);
6. the block means and the two uncertainty estimates;
7. all written answers and the final synthesis.
